# Demonstration of Possible Results


# TODO

```py

# create a proper dashboard with ability to use filters like in Wyscout's advanced search

# for the top N (N=15 default) players, give comparative analysis of their stats, such as market value, and potential resale value

# dashboard should be self contained and only require installs on the developer side, the end user should not have to install anything
# The dashboard should be able to run on a local server and be accessible via a web browser.

```


This notebook will be dedicated to the development of an initial template, for demonstration purposes. We were tasked with deveoping a product with a dashboard-like output.

Our goal for this session will be to setup a structured report generation system. In order to do so, we want the following pipeline to be strict and fully functional:

 1. fetching all player data
 2. preserve all identification structural elements, such as country, league and name (for and disambuguation purposes) 
 3. make use of relevant first-pull information (currently manual, API is not applicable nor is webscrape) to define proper relevance metrics
 4. develop techniques to display information in meaningful, comparative plots
 5. define a dashboard-like structure, with filtering capabilities for convenience
 6. add a pdf output functionality to expedite report quality and sharing 

## Frameworks & Libraries

We'll be using standard DS and graph plotting libraries for this demo.

In [ ]:
# read ../data/2025-26 folder. This contains multiple subfolders, which can contain csv files. If they do contain csv, use them, flag otherwise.
import re
import numpy as np
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

## Getting Working Data

Fetch all `.csv`:

In [2]:
SEASON = "2025-2026"                   
DATA_DIR = Path("../data") / SEASON      

frames, flagged = [], []
for sub in sorted([d for d in DATA_DIR.iterdir() if d.is_dir()]):
    csvs = list(sub.glob("*.csv"))
    if not csvs:

        # flags and skips folders with no csv
        flagged.append(sub.name)         
        continue
    f = pd.read_csv(csvs[0])

    # pulls country code and league from folder name
    m = re.search(r"\[([A-Za-z]{2,3})\]\s*-\s*\((.+)\)", sub.name)   
    f["Country"] = m.group(1) if m else ""
    f["League"]  = m.group(2) if m else sub.name
    frames.append(f)

players = pd.concat(frames, ignore_index=True)
print(f"{len(players)} players across {players['League'].nunique()} leagues")
if flagged:
    print("FLAGGED (no csv, skipped):")
    for name in flagged:
        print("  -", name)

39650 players across 68 leagues
FLAGGED (no csv, skipped):
  - 36 - [CRO] - (First NL)
  - 41 - [USA] - (Major League Soccer)
  - 42 - [USA] - (USL Championship)
  - 60 - [RSA] - (Premier Division)
  - 63 - [ALG] - (Ligue 1)


## Performance & Potential Indicators

<center>

*The core objective of identifying soccer players for profitable resale (the "buy-low, sell-high" trading model)*

</center>

<center>

*relies on **decoupling a player's underlying talent from their current market valuation**.*

</center>

We didn't want to delegate this responsibility to a simple AI chatbot. From the previous reports (given to us as example), we identified clear signs of "drop info and trust output" for report building and reasoning.

In order to get industry certified techniques, we decided to carry out an investigative process and research actual published evidence. This will also allow us to have more robust control of the AI's training process and quality of output, defining a sort of "ground truth" to aspire to reach.

### Satistically Indicative Features for Player Value

According to a study performed by [AMSTATNEWS](https://magazine.amstat.org/) on *"a global dataset of 5,682 professional players and their ratings from FIFA analysts"*, the *"**reactions**, **composure**, **short** **passing**, and **long** **passing** are **significantly correlated with value**, with **technical skills** being **consistently more important than raw physical ability**"*.

#### Feature Correlation & Player Quality Comparison

<center>

![](imgs_demo/correlation.png)
![](imgs_demo/comparison.png)

</center>

### A Study on Performance Metrics as Estimators for Market Value

When considering the study performed by [Yalçınkaya & Işık (2024)](https://doi.org/10.54141/psbd.1489554) in the *Pamukkale Journal of Sport Sciences* on *"1,508 players across Europe's top five leagues, pairing 28 WhoScored performance metrics with Transfermarkt market values"*, the *"**total goals**, **shots per game**, **assists**, **through balls**, **dribbles**, and **key passes** are the **strongest positive correlates of market value**, with **attacking and creative output being consistently more predictive than defensive workload**"*. 

A **Random Forest** model reached an **R² of 0.90**, while **age** carried the **strongest negative correlation (−0.27)**, indicating that younger players are the most preferable.

## Research Conclusions

From the dedicated studies above, we have separated all traits and skills by descending priority, for each role, and **matching them to their Wyscout equivalent** (when possible).

### Role-Independent Features

 1. **Goal output**
    1. Wyscout: combination of `Non-penalty goals per 90`, `Goals per 90`, `xG per 90`
 2. **Shot volume & threat**
    1. Wyscout: `Shots per 90`, `Shots on target, %`
 2. **Chance creation / assists**
    3. Wyscout: `Assists per 90`, `xA per 90`
 3. **Line-breaking passing / through balls**
    1. Wyscout: `Through passes per 90`, `Accurate through passes, %`
 4. **Ball-carrying / dribbling**
    1. Wyscout: `Dribbles per 90`, `Successful dribbles, %`, `Progressive runs per 90`
 5. **Key passing** 
    1. Wyscout: `Key passes per 90`
 6. **Passing volume & accuracy (short & long)
    1. Wyscout: `Passes per 90`, `Accurate passes, %`, `Accurate short / medium passes, %`, `Accurate long passes, %`
 7. **Playing time / reliability**
    1. Wyscout: `Minutes played`, `Matches played`
 8.  **Drawing fouls (being targeted)**
     1.  Wyscout: `Fouls suffered per 90`
 9.  **Age**: considered to be the decoupling variable, with *negative* against current value, *positive* for resale, so it enters the value model and the resale weighting with opposite signs. 
     1.  Wyscout: `Age`

TODO: Role based is giving issues due to underrepresentation!

## Feature Setup: Research-Based Indicators of Potential

In [ ]:
# map Wyscout primary position code -> role bucket
ROLE = {}
for p in ["GK"]:                         ROLE[p] = "GK"
for p in ["CB","RCB","LCB"]:             ROLE[p] = "CB"
for p in ["RB","LB","RWB","LWB"]:        ROLE[p] = "FB"
for p in ["DMF","RDMF","LDMF"]:          ROLE[p] = "DM"
for p in ["CMF","RCMF","LCMF"]:          ROLE[p] = "CM"
for p in ["AMF"]:                        ROLE[p] = "AM"
for p in ["LW","RW","LWF","RWF","LAMF","RAMF"]: ROLE[p] = "W"
for p in ["CF"]:                         ROLE[p] = "CF"

# ROLE QUALITY — percentile these WITHIN the role to score "how good at the job"
ROLE_METRICS = {
 "GK": ["Save rate, %", "Prevented goals per 90", "Conceded goals per 90",
        "Exits per 90", "Accurate passes, %", "Accurate long passes, %"],
 "CB": ["Defensive duels won, %", "Aerial duels won, %", "PAdj Interceptions",
        "Shots blocked per 90", "Successful defensive actions per 90",
        "Accurate passes, %", "Accurate progressive passes, %"],
 "FB": ["Defensive duels won, %", "PAdj Interceptions", "Successful defensive actions per 90",
        "Crosses per 90", "Accurate crosses, %", "Progressive runs per 90",
        "xA per 90", "Key passes per 90"],
 "DM": ["PAdj Interceptions", "Defensive duels won, %", "Successful defensive actions per 90",
        "Accurate passes, %", "Accurate forward passes, %", "Progressive passes per 90",
        "Received passes per 90"],
 "CM": ["Progressive passes per 90", "Accurate passes, %", "Key passes per 90",
        "xA per 90", "Progressive runs per 90", "Passes to final third per 90",
        "Duels won, %"],
 "AM": ["xA per 90", "Key passes per 90", "Through passes per 90", "Shot assists per 90",
        "Passes to penalty area per 90", "Dribbles per 90", "Successful dribbles, %",
        "Touches in box per 90", "xG per 90"],
 "W":  ["Successful dribbles, %", "Progressive runs per 90", "Accelerations per 90",
        "xA per 90", "Key passes per 90", "Crosses per 90", "Accurate crosses, %",
        "xG per 90", "Touches in box per 90", "Fouls suffered per 90"],
 "CF": ["Non-penalty goals per 90", "xG per 90", "Goal conversion, %", "Shots on target, %",
        "Touches in box per 90", "Head goals per 90", "Aerial duels won, %",
        "Fouls suffered per 90"],
}

# VALUE DRIVERS — the paper's selected set (Wyscout equivalents). Feed to the value-gap model.
# Ranked within role too, so a CB isn't punished for scoring less than a CF.
VALUE_FEATURES = ["Age", "Goals per 90", "Shots per 90", "Assists per 90", "xA per 90",
                  "Through passes per 90", "Dribbles per 90", "Successful dribbles, %",
                  "Key passes per 90", "Passes per 90", "Accurate passes, %",
                  "Fouls suffered per 90", "Minutes played"]